## Day 3 - Part 1: RNN 기초: 시간을 기억하는 신경망

### 개요

Day 2에서 우리는 표(tabular) 형태의 데이터를 위한 심층 신경망(DNN)과, 이미지의 공간적 구조를 이해하는 합성곱 신경망(CNN)을 정복했습니다. 

하지만 세상의 데이터는 정적인 표나 이미지뿐만이 아닙니다. 

"오늘 날씨가 좋으니 산책가자" 라는 문장, 어제와 오늘의 주가 흐름, 심장의 박동 소리와 같이 `순서(sequence)` 가 중요한 데이터는 어떻게 처리해야 할까요?

지금까지 배운 DNN이나 CNN에 문장을 단어별로 잘라 넣는다고 상상해 보세요. 

"not happy"와 "happy not"은 완전히 다른 의미를 갖지만, 순서 정보를 고려하지 못하는 모델에게는 그저 같은 단어들의 집합일 뿐입니다. 

데이터에 내재된 시간적, 순차적 맥락을 이해하지 못하는 한계에 부딪히게 됩니다.

이러한 순서가 있는 데이터, 즉 `시퀀스 데이터(Sequence Data)` 를 처리하기 위해 탄생한 것이 바로 `순환 신경망(Recurrent Neural Network, RNN)` 입니다. 

RNN은 마치 '기억'을 가진 신경망처럼, 이전 단계의 정보를 현재 단계의 입력과 함께 처리하여 시간의 흐름 속에서 맥락을 파악합니다.

이번 파트에서는 시퀀스 데이터를 '읽고' '이해'할 수 있는 RNN의 세계로 떠납니다. 신경망이 어떻게 과거를 기억하는지, 그 기억력이 어떤 한계를 가지며, 이를 어떻게 극복하는지(LSTM, GRU) 배우고, 직접 시계열 데이터를 예측하는 모델을 만들어 보겠습니다.

`이번 파트의 학습 목표:`

  * 시퀀스 데이터의 개념과 기존 DNN/CNN의 한계를 이해합니다.
  
  * RNN의 기본 구조와 '기억'의 핵심인 `은닉 상태(Hidden State)` 의 역할을 설명할 수 있습니다.
  * RNN의 학습 원리인 `시간에 따른 역전파(BPTT)` 와 이로 인해 발생하는 `장기 의존성 문제(Long-term Dependency Problem)` 를 이해합니다.
  * RNN의 한계를 극복하기 위해 등장한 `LSTM`과 `GRU`의 핵심 아이디어(게이트 메커니즘)를 설명할 수 있습니다.
  * PyTorch를 사용하여 `nn.RNN`, `nn.LSTM`, `nn.GRU` 레이어로 시퀀스 모델을 구현할 수 있습니다.
  * 시계열 예측 문제를 정의하고, 직접 구축한 RNN 모델로 미래 값을 예측하고 그 결과를 시각화할 수 있습니다.

### 1. 왜 순서가 중요할까?: 시퀀스 데이터의 등장

RNN을 배우기 전, 왜 순서가 중요한지, 그리고 왜 기존 모델로는 부족한지 명확히 짚고 넘어가겠습니다.

  * `시퀀스 데이터란?` 데이터 포인트들이 특정 순서로 배열되어 있으며, 이 순서가 데이터의 의미에 중요한 영향을 미치는 데이터를 말합니다.
      
      * `자연어`: 문장을 구성하는 단어의 순서 (예: "고양이가 쥐를 쫓는다" vs "쥐가 고양이를 쫓는다")
      
      * `시계열 데이터`: 주식 가격, 날씨, 센서 측정값 등 시간의 흐름에 따라 기록된 데이터
      * `음성 및 음악`: 시간의 흐름에 따른 소리의 파형
  
  * `기존 모델의 한계`
      
      * `DNN (완전연결 신경망)`: 입력을 1차원 벡터로 취급하므로, 순서 정보를 완전히 파괴합니다.
      
      * `CNN (합성곱 신경망)`: 2D/3D 공간에서의 지역적 특징을 추출하는 데 특화되어 있어, 시간적/순차적 연속성을 모델링하기에는 적합하지 않습니다.

RNN은 이 문제를 '순환'이라는 천재적인 아이디어로 해결합니다. 

각 타임스텝(time step)에서 입력을 처리한 결과를 다음 타임스텝으로 넘겨주어, 마치 문장을 앞에서부터 차례로 읽으며 맥락을 파악하는 우리 뇌의 작동 방식과 유사하게 동작합니다.

### 2. RNN의 기본 구조: 기억의 비밀, 은닉 상태(Hidden State)

RNN의 핵심은 바로 `순환(recurrent)` 하는 구조에 있습니다. 

이 구조를 통해 과거의 정보를 `은닉 상태(hidden state)` 라는 형태로 저장하고, 현재의 입력을 해석하는 데 사용합니다.

<img src="https://thebook.io/img/080289/372.jpg" width="600">

  * `은닉 상태` (Hidden State, $h_t$)
    *  RNN의 '기억'을 담당하는 핵심 요소입니다. 
    
    *  현재 시점 $t$의 은닉 상태 $h_t$는 `이전 시점의 은닉 상태` $h_{t-1}$ 와 `현재 시점의 입력` $x_t$ 을 함께 입력받아 계산됩니다. 
    *  수식으로는 $h_t = \tanh(W_{hh}h_{t-1} + W_{xh}x_t + b_h)$ 와 같이 표현할 수 있습니다.
  
  * `가중치 공유` (Weight Sharing)
    *  오른쪽의 펼쳐진 그림에서 모든 타임스텝의 $W$는 사실 모두 같은 가중치입니다. 
    
    *  즉, RNN은 모든 타임스텝에 걸쳐 `동일한 가중치 행렬`($W_{hh}, W_{xh}$) 을 공유합니다. 
    *  이는 학습할 파라미터 수를 크게 줄여주고, 시퀀스 길이에 상관없이 모델을 일반화할 수 있게 해주는 핵심적인 특징입니다.
  
  * `출력` (Output, $y_t$)
    * 각 시점의 출력 $y_t$는 해당 시점의 은닉 상태 $h_t$를 변환하여 계산됩니다. 
    
    * ($y_t = W_{hy}h_t + b_y$)


#### 코드 실습: `nn.RNN`으로 RNN의 입출력 이해하기

PyTorch의 `nn.RNN`을 통해 RNN 레이어의 입출력 형태를 직접 확인해 봅시다.

In [ ]:
import torch
import torch.nn as nn

# 하이퍼파라미터 설정
batch_size = 1     # batch_size == 레코드 수 : 한 번에 처리할 시퀀스 데이터의 수
seq_len = 5        # seq_len == 텍스트 데이터 - 단어 개수 / 다른 데이터 - 순열 개수 : 시퀀스의 길이 (예: 5개 단어)
input_size = 10    # 각 입력 요소의 차원 (예: 단어 벡터의 차원)
hidden_size = 20   # 은닉 상태 벡터의 차원

# nn.RNN 레이어 정의
rnn_layer = nn.RNN(
    input_size=input_size,
    hidden_size=hidden_size,
    batch_first=True  # 입력 텐서의 첫 번째 차원을 배치 크기로 설정
)

# (배치 크기, 시퀀스 길이, 입력 크기) 형태의 더미 입력 데이터 생성
dummy_inputs = torch.randn(batch_size, seq_len, input_size)
# 3d - 데이터를 3차원으로 만들어줌

# 초기 은닉 상태 (입력하지 않으면 0으로 초기화됨)
# (층의 수, 배치 크기, 은닉 크기)
initial_hidden_state = torch.zeros(1, batch_size, hidden_size)

# RNN 연산 수행
# outputs: 모든 시점(t=1~5)의 은닉 상태를 모은 것
# last_hidden_state: 마지막 시점(t=5)의 은닉 상태
outputs, last_hidden_state = rnn_layer(dummy_inputs, initial_hidden_state)

print("원본 입력 형태:", dummy_inputs.shape)
print("모든 출력 형태:", outputs.shape)           # (배치 크기, 시퀀스 길이, 은닉 크기)
print("마지막 은닉 상태 형태:", last_hidden_state.shape) # (층의 수, 배치 크기, 은닉 크기)

원본 입력 형태: torch.Size([1, 5, 10])
모든 출력 형태: torch.Size([1, 5, 20])
마지막 은닉 상태 형태: torch.Size([1, 1, 20])


`outputs`는 시퀀스의 매 순간순간에 대한 모델의 '생각(은닉 상태)'을 담고 있으며, 

`last_hidden_state`는 전체 시퀀스를 모두 읽고 난 후의 최종적인 '결론(마지막 은닉 상태)'이라고 할 수 있습니다.


### 3. RNN의 학습과 한계: 장기 기억상실증 문제

RNN은 `순서(sequence)` 라는 방식으로 학습합니다. 

이는 시간 순으로 펼쳐진 네트워크에 일반적인 역전파 알고리즘을 적용하는 것과 같습니다. 

마지막 시점의 출력에서 발생한 오차(loss)를 역방향으로 전파시켜 모든 시점에 공유되는 가중치 $W$를 업데이트합니다.

하지만 이 과정에서 RNN은 치명적인 한계에 부딪힙니다.

#### 3.1. 기울기 소실과 기울기 폭발 문제

* `시퀀스 데이터(Sequence Data)`
  
  * 시퀀스가 길어질수록, BPTT(Backpropagation Through Time) 과정에서 기울기가 역방향으로 전파되면서,
  
  * 너무 작아지거나(기울기 소실, Vanishing Gradient) 
  * 너무 커지는(기울기 폭발, Exploding Gradient) 현상이 발생합니다.

#### 3.2. 장기 의존성 문제(Long-term Dependency Problem)

<img src="https://i.imgur.com/H9UoXdC.png" width="500">

* 시퀀스의 앞부분에 있던 중요한 정보에 대한 기울기가 거의 0에 가까워져 해당 정보로부터 모델이 거의 학습하지 못하게 됩니다. 

* 이는 마치 RNN이 '장기 기억상실증'에 걸린 것처럼, 문장의 맨 앞에 있던 핵심 단어를 문장 끝에서는 잊어버리는 것과 같습니다.

#### 3.3. 문제의 원인

RNN의 이러한 한계는 주로 다음과 같은 이유로 발생합니다:

1. `반복적인 가중치 곱셈`: 시퀀스가 길어질수록 같은 가중치 행렬을 여러 번 곱하게 되어, 고유값이 1보다 크면 기울기 폭발, 1보다 작으면 기울기 소실이 발생합니다.

2. `제한된 정보 흐름`: 단순한 은닉 상태만으로는 복잡한 장기 의존성을 효과적으로 기억하기 어렵습니다.

3. `고정된 기억 용량`: RNN의 은닉 상태 크기가 고정되어 있어, 중요한 정보와 덜 중요한 정보를 구분하여 저장할 수 없습니다.

### 4. 기억력 향상시키기: LSTM과 GRU

이러한 RNN의 '기억상실증' 문제를 해결하기 위해, 

더욱 정교한 기억 장치를 갖춘 `LSTM(Long Short-Term Memory)`과 `GRU(Gated Recurrent Unit)`가 등장했습니다. 

이 둘의 핵심 아이디어는 `게이트 메커니즘(Gate Mechanism)`을 도입하여, 기억할 정보와 잊어버릴 정보를 효과적으로 제어하는 것입니다.

<img src="https://miro.medium.com/v2/resize:fit:1200/1*I5iwCL8zDo9OppBPsprwTw.png" width="800">


#### 4.1. LSTM (Long Short-Term Memory): 정교한 기억 관리 시스템

LSTM은 RNN의 은닉 상태 외에, 장기 기억을 위한 `셀 상태(Cell State)` 라는 별도의 통로를 가지고 있습니다. 

이 셀 상태를 통해 정보가 얼마나 오래 보존될지 3개의 게이트가 관리합니다.

1.  `망각 게이트(Forget Gate)`: 과거의 정보($c_{t-1}$) 중 어떤 것을 잊어버릴지 결정합니다.

2.  `입력 게이트(Input Gate)`: 현재 정보($x_t$) 중 어떤 것을 셀 상태에 새로 저장할지 결정합니다.
3.  `출력 게이트(Output Gate)`: 셀 상태의 정보 중 어떤 것을 현재의 은닉 상태($h_t$)로 내보낼지 결정합니다.

이러한 정교한 제어 시스템 덕분에 LSTM은 시퀀스가 매우 길어져도 중요한 정보를 잃지 않고 오랫동안 기억할 수 있습니다.

#### 4.2. GRU (Gated Recurrent Unit): 효율적인 경량 기억 장치

GRU는 LSTM의 구조를 단순화한 모델입니다. 별도의 셀 상태 없이 은닉 상태만으로 기억을 관리하며, 게이트의 수도 2개로 줄였습니다.

1.  `리셋 게이트(Reset Gate)`: 과거의 기억을 얼마나 무시할지 결정합니다.

2.  `업데이트 게이트(Update Gate)`: 과거의 기억과 현재 정보 중 어느 것을 어느 비율로 섞어 새로운 기억을 만들지 결정합니다. (LSTM의 망각+입력 게이트 역할)

GRU는 LSTM보다 파라미터 수가 적어 계산이 빠르면서도, 많은 경우에 LSTM과 비슷한 성능을 보여줍니다. 

데이터가 적거나 빠른 학습이 필요할 때 좋은 대안이 될 수 있습니다.

#### 코드 실습: `nn.LSTM`과 `nn.GRU` 사용하기

PyTorch에서는 `nn.RNN`을 `nn.LSTM`이나 `nn.GRU`로 간단히 교체하여 사용할 수 있습니다.

In [ ]:
# LSTM 레이어 정의
lstm_layer = nn.LSTM(
    input_size=input_size,
    hidden_size=hidden_size,
    batch_first=True
)

# LSTM은 (마지막 은닉 상태, 마지막 셀 상태)를 튜플로 반환
# last_cell_lstm : RNN과 달리 마지막 셀 상태도 반환 
outputs_lstm, (last_hidden_lstm, last_cell_lstm) = lstm_layer(dummy_inputs)
print("LSTM 마지막 은닉 상태 형태:", last_hidden_lstm.shape)
print("LSTM 마지막 셀 상태 형태:", last_cell_lstm.shape)

LSTM 마지막 은닉 상태 형태: torch.Size([1, 1, 20])
LSTM 마지막 셀 상태 형태: torch.Size([1, 1, 20])


In [3]:
# GRU 레이어 정의
gru_layer = nn.GRU(
    input_size=input_size,
    hidden_size=hidden_size,
    batch_first=True
)

# GRU는 마지막 은닉 상태만 반환
outputs_gru, last_hidden_gru = gru_layer(dummy_inputs)
print("GRU 마지막 은닉 상태 형태:", last_hidden_gru.shape)

GRU 마지막 은닉 상태 형태: torch.Size([1, 1, 20])


LSTM은 은닉 상태 외에 셀 상태를 추가로 반환하는 점이 다릅니다. 

이 외에는 기본적인 사용법이 거의 동일하여 쉽게 교체하며 실험할 수 있습니다.

### 5. 종합 실습: 일일 최저 기온 예측하기

이제 배운 내용을 총동원하여, 호주 멜버른의 일일 최저 기온 데이터를 사용하여 내일의 기온을 예측하는 시계열 모델을 만들어 보겠습니다.

#### 5.1. 데이터 준비 및 전처리

먼저, Open-Meteo Free API를 이용해서 기온 데이터를 불러와 시각화하고, 모델이 학습할 수 있는 형태로 가공합니다.

In [4]:
import requests
import pandas as pd
from datetime import datetime, timedelta
import plotly.express as px

# Open-Meteo API를 사용하여 멜버른의 1년치 기온 데이터 가져오기
def get_melbourne_weather_data():
    # 멜버른의 좌표 (위도: -37.8136, 경도: 144.9631)
    latitude = -37.8136
    longitude = 144.9631
    
    # 1년 전부터 오늘까지의 날짜 범위 설정
    end_date = datetime.now()
    start_date = end_date - timedelta(days=365)
    
    # Open-Meteo API URL
    url = "https://archive-api.open-meteo.com/v1/archive"
    
    params = {
        "latitude": latitude,
        "longitude": longitude,
        "start_date": start_date.strftime("%Y-%m-%d"),
        "end_date": end_date.strftime("%Y-%m-%d"),
        "daily": "temperature_2m_min,temperature_2m_max",
        "timezone": "Australia/Melbourne"
    }
    
    response = requests.get(url, params=params)
    response.raise_for_status()
    data = response.json()
    # 데이터프레임으로 변환
    df = pd.DataFrame({
        'date': pd.to_datetime(data['daily']['time']),
        'temp_min': data['daily']['temperature_2m_min'],
        'temp_max': data['daily']['temperature_2m_max']
    })
    
    # 날짜를 인덱스로 설정
    df.set_index('date', inplace=True)
    
    return df


In [5]:
# 1. 데이터 가져오기
melbourne_df = get_melbourne_weather_data()
melbourne_df.head()

,temp_min,temp_max
date,,
2024-06-25,8.7,14.7
2024-06-26,7.9,13.8
2024-06-27,7.2,14.6
2024-06-28,7.4,14.9
2024-06-29,7.5,13.8


In [6]:
pd.options.plotting.backend = "plotly"

In [7]:
melbourne_df.plot(title='멜버른 일일 기온 범위 (최근 1년)', labels={'value': '기온 (°C)', 'variable': '기온 유형'})

In [8]:
import numpy as np
from sklearn.preprocessing import MinMaxScaler

# 2. 데이터 스케일링 (0~1 사이 값으로 정규화)
scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(melbourne_df['temp_max'].values.reshape(-1, 1))

In [9]:
# 3. 시퀀스 데이터 생성
# 과거 30일 데이터(X)로 다음 날의 기온(y)을 예측하는 데이터셋 생성
def create_sequences(data, seq_length):
    xs, ys = [], []
    for i in range(len(data) - seq_length):
        x = data[i:i+seq_length]
        y = data[i+seq_length]
        xs.append(x)
        ys.append(y)
    return np.array(xs), np.array(ys)

SEQ_LENGTH = 30
X, y = create_sequences(scaled_data, SEQ_LENGTH)

In [10]:
# 4. 훈련/테스트 데이터 분리 (마지막 1년치를 테스트 데이터로 사용)
train_size = int(len(X) * 0.9)
X_train, X_test = X[:train_size], X[train_size:] # 시퀀스 데이터는 비율로 슬라이스 / 함수 쓰면 섞일 수 있기 때문
y_train, y_test = y[:train_size], y[train_size:]

In [11]:
# 5. PyTorch 텐서로 변환 및 DataLoader 생성
from torch.utils.data import TensorDataset, DataLoader

X_train_tensor = torch.FloatTensor(X_train)
y_train_tensor = torch.FloatTensor(y_train)
X_test_tensor = torch.FloatTensor(X_test)
y_test_tensor = torch.FloatTensor(y_test)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

#### 5.2. 시계열 예측 모델 정의하기

LSTM을 사용하여 시계열 예측 모델을 정의합니다. 마지막 시점의 은닉 상태를 받아, 완전연결층을 통해 최종 예측값(내일의 기온)을 출력합니다.

In [12]:
class TempPredictor(nn.Module):
    def __init__(self, n_features, hidden_dim, n_layers=1):
        super(TempPredictor, self).__init__()
        self.hidden_dim = hidden_dim
        self.n_layers = n_layers

        # LSTM 레이어
        self.lstm = nn.LSTM(
            input_size=n_features, # 입력 특성 수
            hidden_size=hidden_dim, # 은닉 상태 크기
            num_layers=n_layers, # 레이어 개수
            batch_first=True # 배치 차원을 첫 번째 차원으로 두는 옵션
        )
        # 완전연결층 (출력=1, 내일의 기온)
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        # LSTM의 출력과 은닉 상태를 모두 받음
        lstm_out, (hidden, cell) = self.lstm(x)
        
        # 마지막 시점의 출력을 사용 (batch_first=True이므로 마지막 차원)
        out = lstm_out[:, -1, :]  # (batch_size, hidden_dim)
        out = self.fc(out)
        return out

# 모델 인스턴스 생성
model = TempPredictor(n_features=1, hidden_dim=50)

# torchsummary 대신 모델의 구조를 직접 출력
print("모델 구조:")
print(model)

# 모델의 파라미터 수 계산
total_params = sum(p.numel() for p in model.parameters())
print(f"총 파라미터 수: {total_params:,}")


모델 구조:
TempPredictor(
  (lstm): LSTM(1, 50, batch_first=True)
  (fc): Linear(in_features=50, out_features=1, bias=True)
)
총 파라미터 수: 10,651


#### 5.3. 모델 학습 및 평가

정의한 모델을 훈련 데이터로 학습시킵니다. 회귀 문제이므로 손실 함수는 `MSELoss`를 사용합니다.

In [13]:
import torch.optim as optim

# 손실 함수 및 옵티마이저
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 모델 학습
num_epochs = 20
for epoch in range(num_epochs):
    for seqs, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(seqs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

    if (epoch+1) % 5 == 0:
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')

print("학습 완료!")

Epoch [5/20], Loss: 0.0335
Epoch [10/20], Loss: 0.0196
Epoch [15/20], Loss: 0.0391
Epoch [20/20], Loss: 0.0259
학습 완료!


#### 5.4. 결과 시각화 및 분석

학습된 모델로 테스트 데이터에 대한 예측을 수행하고, 실제 값과 예측 값을 함께 그래프로 그려 성능을 확인합니다.

In [14]:
import plotly.graph_objects as go

model.eval() # 평가 모드
with torch.no_grad():
    test_preds = model(X_test_tensor).numpy()

# 스케일링된 예측값을 원래 스케일로 되돌림
unscaled_preds = scaler.inverse_transform(test_preds)
unscaled_actuals = scaler.inverse_transform(y_test)

# 시각화
fig = go.Figure()
fig.add_trace(go.Scatter(y=unscaled_actuals.flatten(), name='Actual Temperature'))
fig.add_trace(go.Scatter(y=unscaled_preds.flatten(), name='Predicted Temperature'))
fig.update_layout(title='Temperature Prediction Results', xaxis_title='Time', yaxis_title='Temperature')
fig.show()

그래프를 통해 모델이 실제 기온의 전반적인 추세를 잘 따라 예측하는 것을 확인할 수 있습니다. 

이것이 바로 시퀀스 데이터의 맥락을 '기억'하고 학습한 RNN 계열 모델의 힘입니다.